In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import  seaborn as sns


In [ ]:
train_df=pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv' , parse_dates=['date'])
test_df=pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv' , parse_dates=['date'])
stores_df=pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')
oil_df=pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/oil.csv' ,parse_dates=['date'])
holidays=pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv' ,parse_dates=['date'])
transactions=pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/transactions.csv' ,parse_dates=['date'])


In [ ]:
print(train_df.info())
print(train_df.describe())
print(train_df['store_nbr'].nunique() , "Stores")
print(train_df['family'].nunique() , "product family")



**Sample time series plot**

In [ ]:
sample=train_df[(train_df['store_nbr'] == 1) &(train_df['family']=='GROCERY I')]

sample.set_index('date')['sales'].plot(figsize=(10,4), title="sales for store 1")
plt.show()

# **Feature Engineering**

In [ ]:
train_df['day'] = train_df['date'].dt.day
train_df['month'] = train_df['date'].dt.month
train_df['year'] = train_df['date'].dt.year
train_df['dayofweek'] = train_df['date'].dt.dayofweek
train_df['weekofyear'] = train_df['date'].dt.isocalendar().week
train_df['is_weekend'] = train_df['dayofweek'] >= 5


**Lag Features**

In [ ]:
train_df['sales_lag7']=train_df.groupby(['store_nbr','family'])['sales'].shift(7)
train_df['rolling_mean_7']=train_df.groupby(['store_nbr','family'])['sales'].shift(1).rolling(7).mean()

**Merging External Data**

In [ ]:
train_df=train_df.merge(stores_df,on='store_nbr' , how='left')
train_df=train_df.merge(oil_df,on='date' , how='left')
train_df=train_df.merge(transactions,on=['date','store_nbr'] , how='left')


#filter Holiday events to ecuador and relevant Dates
train_df = train_df.merge(holidays[['date', 'type']], on='date', how='left', suffixes=('', '_holiday'))
train_df['is_holiday'] = train_df['type_holiday'].notna().astype(int)

# EDA

In [ ]:
print(train_df['date'].min(), '→', train_df['date'].max())
print('Number of stores:', train_df['store_nbr'].nunique())
print('Number of families:', train_df['family'].nunique())
print('Any missing values?\n', train_df.isnull().sum())


**Total sales over time**

In [ ]:
daily_sales = train_df.groupby('date')['sales'].sum()
daily_sales.plot(figsize=(15,4), title='Total Sales Over Time')
plt.ylabel('Sales')
plt.xlabel('Date')
plt.grid()
plt.show()

**sales by day of week**

In [ ]:
train_df['dayofweek'] = train_df['date'].dt.dayofweek
dow_avg = train_df.groupby('dayofweek')['sales'].mean()

plt.figure(figsize=(7,4))
dow_avg.plot(kind='bar', title='Average Sales by Day of Week')
plt.xticks(ticks=range(7), labels=['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], rotation=0)
plt.ylabel('Avg Sales')
plt.show()


**Sales on Holidays vs Non-Holidays**

In [ ]:
sns.boxplot(data=train_df, x='is_holiday', y='sales')
plt.xticks([0,1], ['Non-Holiday', 'Holiday'])
plt.title('Sales on Holidays vs Non-Holidays')
plt.show()

**Sales Trends for Sample Store/Family**

In [ ]:
sample = train_df[(train_df['store_nbr'] == 1) & (train_df['family'] == 'GROCERY I')]
sample.set_index('date')['sales'].plot(figsize=(15,4), title='Store 1 - GROCERY I Sales Trend')
plt.ylabel('Sales')
plt.show()

Transactions vs Sales Correlation

In [ ]:
sns.scatterplot(data=train_df, x='transactions', y='sales', alpha=0.3)
plt.title('Transactions vs Sales')
plt.show()


**Feature Engineering**

**Lag Features**

In [ ]:
train_df = train_df.sort_values(by=['store_nbr', 'family', 'date'])

for lag in [7, 14, 28]:
    train_df[f'sales_lag_{lag}'] = train_df.groupby(['store_nbr', 'family'])['sales'].shift(lag)


**Rolling Window Features**

In [ ]:
for window in [7, 14, 30]:
    train_df[f'sales_roll_mean_{window}'] = train_df.groupby(['store_nbr', 'family'])['sales'].shift(1).rolling(window).mean()


**Encode Categorical columns**

In [ ]:
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()
train_df['family_enc']=le.fit_transform(train_df['family'])
train_df['store_enc']=le.fit_transform(train_df['store_nbr'])




**Droping nulls**

In [ ]:
train_df.dropna(inplace=True)


# **Modeling**

In [ ]:
#Dropping columns not needed
features = [
    'store_enc', 'family_enc', 'day', 'month', 'year',
    'dayofweek', 'weekofyear', 'is_weekend', 'is_holiday',
    'sales_lag_7', 'sales_lag_14', 'sales_lag_28',
    'sales_roll_mean_7', 'sales_roll_mean_14', 'sales_roll_mean_30'
]
target='sales'

#Drop missing values
model_data=train_df.dropna(subset=features+[target])

In [ ]:
from sklearn.model_selection import train_test_split

split_date='2017-06-01'
train_set=model_data[model_data['date']<split_date]
valid_set=model_data[model_data['date']>=split_date]


X_train,y_train=train_set[features],train_set[target]
X_valid,y_valid=valid_set[features],valid_set[target]


**Train Models**

**Random Forest**

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_log_error

rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_valid)

rf_rmsle = np.sqrt(mean_squared_log_error(y_valid, np.maximum(0, rf_preds)))
print("📌 Random Forest RMSLE:", rf_rmsle)

**XGBoost**

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, objective='reg:squarederror', random_state=42)
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_valid)

xgb_rmsle = np.sqrt(mean_squared_log_error(y_valid, np.maximum(0, xgb_preds)))
print("📌 XGBoost RMSLE:", xgb_rmsle)


**LightGBM**

In [ ]:
import lightgbm as lgb

lgbm = lgb.LGBMRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42)
lgbm.fit(X_train, y_train)
lgbm_preds = lgbm.predict(X_valid)

lgbm_rmsle = np.sqrt(mean_squared_log_error(y_valid, np.maximum(0, lgbm_preds)))
print("📌 LightGBM RMSLE:", lgbm_rmsle)


**Visualizing Predections**

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15,4))
plt.plot(y_valid.values[:100], label='True')
plt.plot(lgbm_preds[:100], label='Predicted')
plt.title("Sales Prediction (First 100 examples)")
plt.legend()
plt.show()
